# 🏷️ Notebook 02 — Sentiment Classification

Fine-tune and evaluate three architectures on Amharic 3-class sentiment:
- **mBERT-Amharic** (encoder-only, ~110 M)
- **mT5-small** (encoder-decoder, ~300 M)
- **BLOOMZ-560M** (decoder-only, ~560 M)

Each model is fine-tuned with **LoRA** (r=8, α=16) for 5 epochs.
Metrics: accuracy, macro-F1, per-class P/R/F1, ECE, confusion matrix.

In [ ]:
# ── Install (Colab) ──
import importlib, subprocess, sys
for pkg in ['torch', 'transformers', 'peft', 'datasets', 'accelerate',
            'scikit-learn', 'sentencepiece', 'sacrebleu', 'bert-score',
            'codecarbon', 'pyyaml', 'nvidia-ml-py']:
    if not importlib.util.find_spec(pkg.replace('-', '_')):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO = Path('.').resolve()
if REPO.name == 'notebooks': REPO = REPO.parent
sys.path.insert(0, str(REPO)); os.chdir(REPO)

from src.utils import load_config, set_all_seeds, MODEL_KEYS, ARCH_LABEL
cfg = load_config()
SEED = cfg['experiment']['seeds'][0]
set_all_seeds(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Prepare Data (if not already done)

In [ ]:
data_dir = Path(cfg['experiment']['processed_data_dir'])
if not (data_dir / 'manifest.json').exists():
    from src.preprocessing.data_prep import prepare_all
    prepare_all(cfg, SEED)
train_df = pd.read_csv(data_dir / 'sentiment_train.csv')
val_df   = pd.read_csv(data_dir / 'sentiment_val.csv')
labels = cfg['datasets']['sentiment']['labels']
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Labels: {labels}')

## 2. Training Loop (per model)

We iterate over the three models. Each run:
1. Loads the pretrained checkpoint + tokenizer
2. Applies LoRA (r=8, α=16)
3. Trains for 5 epochs with weighted CE loss
4. Evaluates and saves results

In [ ]:
from src.training.sentiment import run_sentiment

results = {}
condition = 'lora'  # change to run other conditions: probe, lora_cda, lora_cda_rw

for model_key in MODEL_KEYS:
    print(f'\n{"="*60}\n  {ARCH_LABEL[model_key]} — {cfg["models"][model_key]["hf_id"]}\n{"="*60}')
    try:
        out_dir = run_sentiment(cfg, model_key, condition, SEED)
        results[model_key] = {
            'metrics': json.loads((out_dir / 'metrics.json').read_text()),
            'efficiency': json.loads((out_dir / 'efficiency.json').read_text()),
            'predictions': pd.read_csv(out_dir / 'predictions.csv'),
        }
    except Exception as e:
        print(f'  FAILED: {e}')
        results[model_key] = None

## 3. Results Comparison

In [ ]:
rows = []
for m in MODEL_KEYS:
    r = results.get(m)
    if r is None: continue
    met = r['metrics']
    rows.append({'Architecture': ARCH_LABEL[m],
                 'Accuracy': f"{met['accuracy']:.4f}",
                 'Macro-F1': f"{met['macro_f1']:.4f}",
                 'ECE': f"{met['calibration']['ece']:.4f}",
                 **{f'{l} F1': f"{met['per_class'][l]['f1']:.4f}" for l in labels}})
cmp = pd.DataFrame(rows)
print(cmp.to_markdown(index=False))

### 3.1 Accuracy & F1 Bar Chart

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

avail = [m for m in MODEL_KEYS if results.get(m)]
x = np.arange(len(avail))
fig, ax = plt.subplots(figsize=(7, 4))
w = 0.35
ax.bar(x - w/2, [results[m]['metrics']['accuracy'] for m in avail], w, label='Accuracy')
ax.bar(x + w/2, [results[m]['metrics']['macro_f1'] for m in avail], w, label='Macro-F1')
ax.set_xticks(x, [ARCH_LABEL[m] for m in avail])
ax.set_ylim(0, 1)
ax.legend()
ax.set_title('Sentiment: Accuracy and Macro-F1')
plt.tight_layout(); plt.show()

### 3.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(avail), figsize=(4.5*len(avail), 4), squeeze=False)
for ax, m in zip(axes[0], avail):
    cm = np.array(results[m]['metrics']['confusion_matrix'])
    ax.imshow(cm, cmap='Blues')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=10)
    ax.set_xticks(range(len(labels)), labels, fontsize=9)
    ax.set_yticks(range(len(labels)), labels, fontsize=9)
    ax.set_title(ARCH_LABEL[m])
    ax.set_xlabel('predicted'); ax.set_ylabel('gold')
plt.tight_layout(); plt.show()

## 4. Next Steps

- Run **all four bias conditions** by changing `condition` above (probe, lora, lora_cda, lora_cda_rw)
- See Notebook 04 for bias analysis
- See Notebook 06 for statistical significance tests